# Prompt-mode Ablation — Essays Test Set

Runs one of five prompt modes on the Essays test set and displays:
- Per-trait macro-F1 (for `tab:prompt-ablation`)
- Predicted-high ratio vs. ground-truth ratio (for `tab:bias`)
- Retrieval statistics: mean-match-rate, hit-rate, and case distribution (for `tab:retrieval-ablation`)

**Prompt modes** (set `prompt_mode` below):
| `prompt_mode` | Ablation row |
|---|---|
| `reasoned_rag_def_oneshot_30f` | **Reasoned-RAG-Def-Oneshot** (proposed, 30-facet) |

**Requires:** `data/vector_db/essays_dual/` built by `build_dual_index.ipynb`.

## Configuration

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve()
# walk up until we find the project root (contains ptd_model/)
for _ in range(4):
    if (project_root / "ptd_model").exists():
        break
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# ── prompt mode ──────────────────────────────────────────────────────────────
prompt_mode    = "reasoned_rag_def_oneshot_30f"

# ── retrieval ────────────────────────────────────────────────────────────────
top_k          = 5
vector_db_dir  = str(project_root / "data/vector_db/essays_dual")

# ── model ────────────────────────────────────────────────────────────────────
model_name     = "gpt-4o-mini"
profiler_model = "gpt-4o-mini"
max_new_tokens = 1024
temperature    = 0.0

# ── paths ────────────────────────────────────────────────────────────────────
test_csv = str(project_root / "data/split/essays/test.csv")
res_dir  = str(project_root / "result")
log_dir  = str(project_root / "log")

print(f"Project root : {project_root}")
print(f"prompt_mode  : {prompt_mode}")
print(f"top_k        : {top_k}")
print(f"vector_db    : {vector_db_dir}")

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from ptd_model.predict import predict
from ptd_model.evaluate import evaluate

TRAIT_NAMES  = ["Openness", "Conscientiousness", "Extraversion", "Agreeableness", "Neuroticism"]
TRAIT_CODES  = ["cOPN", "cCON", "cEXT", "cAGR", "cNEU"]
TRAIT_RAG    = [
    "Openness to Experience", "Conscientiousness",
    "Extraversion", "Agreeableness", "Neuroticism",
]

test_df = pd.read_csv(test_csv)
print(f"Test set: {len(test_df)} rows")

## Step 1 — Run prediction

In [ ]:
run_id, run_time, prediction_csv = predict(
    text_df        = test_df,
    model_name     = model_name,
    log_dir        = log_dir,
    prompt_mode    = prompt_mode,
    max_new_tokens = max_new_tokens,
    res_dir        = res_dir,
    temperature    = temperature,
    top_k          = top_k,
    vector_db_dir  = vector_db_dir,
    profiler_model = profiler_model,
)

print(f"Done in {run_time:.1f}s")
print(f"Predictions: {prediction_csv}")

## Step 2 — Evaluate (macro-F1)

In [ ]:
evaluation = evaluate(
    prediction_csv = prediction_csv,
    model_name     = model_name,
    res_dir        = res_dir,
    run_time       = run_time,
    prompt_mode    = prompt_mode,
    run_id         = run_id,
)

summary_df = pd.read_csv(evaluation["summary_csv"])
print(f"Failed: {evaluation['fail_count']} / {evaluation['n_records']}")
display(
    summary_df[["trait", "n_samples", "accuracy", "macro_f1", "weighted_f1"]]
    .sort_values("trait")
    .reset_index(drop=True)
)

## Step 3 — Macro-F1 table (thesis `tab:prompt-ablation` row)

In [ ]:
trait_order = ["Openness", "Conscientiousness", "Extraversion", "Agreeableness", "Neuroticism"]
f1_row = {}
for t in trait_order:
    row = summary_df[summary_df["trait"] == t]
    f1_row[t] = round(row["macro_f1"].values[0], 4) if len(row) else None

macro_avg = round(sum(v for v in f1_row.values() if v is not None) / len(trait_order), 4)

result_row = pd.DataFrame([{
    "prompt_mode": prompt_mode,
    **f1_row,
    "Macro avg": macro_avg,
}])
display(result_row)

print("\nLaTeX row (paste into tab:prompt-ablation):")
cells = " & ".join(str(f1_row[t]) for t in trait_order)
print(f"  {prompt_mode} & {cells} & {macro_avg} \\\\")

## Step 4 — Prediction bias analysis (thesis `tab:bias` row)

In [ ]:
pred_df = pd.read_csv(prediction_csv)

gt_ratios   = {}
pred_ratios = {}
for tname, tcode in zip(TRAIT_NAMES, TRAIT_CODES):
    gt_col   = tcode
    pred_col = f"pred_{tcode}"
    if gt_col in test_df.columns:
        gt_ratios[tname] = round(test_df[gt_col].map(lambda x: 1 if str(x).strip().lower() in ("1","1.0","high") else 0).mean(), 4)
    if pred_col in pred_df.columns:
        pred_ratios[tname] = round(pred_df[pred_col].map(lambda x: 1 if str(x).strip().lower() in ("1","1.0","high") else 0).mean(), 4)

bias_df = pd.DataFrame({
    "Ground truth":  [gt_ratios.get(t) for t in TRAIT_NAMES],
    "Predicted high": [pred_ratios.get(t) for t in TRAIT_NAMES],
    "Drift":          [round((pred_ratios.get(t,0) - gt_ratios.get(t,0)), 4) for t in TRAIT_NAMES],
}, index=TRAIT_NAMES)
display(bias_df)

print("\nLaTeX row (paste into tab:bias):")
cells = " & ".join(str(pred_ratios.get(t,"")) for t in TRAIT_NAMES)
print(f"  {prompt_mode} & {cells} \\\\")

## Step 5 — Retrieval statistics

Computes per-query label-match counts across all traits, then reports:
- **Mean-match-rate (MMR)**: avg fraction of top-k retrieved with same label
- **Hit-rate (HR)**: fraction of queries with at least one matching label
- **Case distribution**: how many queries had 0 / 1 / ... / k same-label retrievals
- **Accuracy by case**: how prediction accuracy correlates with retrieval quality

In [ ]:
import rag.retriever as _retriever_mod
from rag.retriever import FeatureRAGRetriever

retriever = FeatureRAGRetriever(db_dir=vector_db_dir)

# ── collect retrieval records ─────────────────────────────────────────────────
records = []  # {trait, query_idx, n_match, query_label}
for idx, row in test_df.iterrows():
    text = row["text"]
    for tname, tcode, trag in zip(TRAIT_NAMES, TRAIT_CODES, TRAIT_RAG):
        gt_val = str(row.get(tcode, "")).strip().lower()
        query_label = "high" if gt_val in ("1", "1.0", "high") else "low"

        retrieved = retriever.retrieve(posts=text, trait=trag, top_k=top_k)
        labels = [r.get("label", "").strip().lower() for r in retrieved]
        n_match = sum(1 for l in labels if l == query_label)

        records.append({
            "trait":       tname,
            "query_idx":   idx,
            "query_label": query_label,
            "n_match":     n_match,
            "k":           len(labels),
        })
    if (idx + 1) % 50 == 0:
        print(f"  Retrieval stats: {idx+1}/{len(test_df)} done")

rag_df = pd.DataFrame(records)
print(f"Total retrieval records: {len(rag_df)}")

In [ ]:
# ── per-trait MMR and HR ──────────────────────────────────────────────────────
mmr_rows = []
for tname in TRAIT_NAMES:
    t = rag_df[rag_df["trait"] == tname]
    mmr = (t["n_match"] / t["k"]).mean()
    hr  = (t["n_match"] >= 1).mean()
    mmr_rows.append({"Trait": tname, "MMR": round(mmr, 4), "HR": round(hr, 4)})

mmr_df = pd.DataFrame(mmr_rows)
mmr_df.loc[len(mmr_df)] = {
    "Trait":  "Macro avg",
    "MMR":    round(mmr_df["MMR"].mean(), 4),
    "HR":     round(mmr_df["HR"].mean(), 4),
}
print(f"Retrieval stats  |  strategy: {prompt_mode}  |  k={top_k}")
display(mmr_df)

print("\nLaTeX row (paste into tab:retrieval-ablation):")
for row in mmr_df[mmr_df["Trait"] != "Macro avg"].itertuples():
    print(f"  {row.Trait}: MMR={row.MMR}, HR={row.HR}")

In [ ]:
# ── overall case distribution ─────────────────────────────────────────────────
pred_df2 = pd.read_csv(prediction_csv)

# map predictions to match query labels for accuracy-by-case
def pred_correct(row, tname, tcode):
    pred_col = f"pred_{tcode}"
    gt_val   = str(row.get(tcode, "")).strip().lower()
    gt_label = "high" if gt_val in ("1", "1.0", "high") else "low"
    pred_val = str(row.get(pred_col, "")).strip().lower()
    pred_label = "high" if pred_val in ("1", "1.0", "high") else "low"
    return int(gt_label == pred_label)

# merge predictions into rag_df
pred_lookup = {}
for tname, tcode in zip(TRAIT_NAMES, TRAIT_CODES):
    for _, prow in pred_df2.iterrows():
        pred_lookup[(int(prow.name), tname)] = pred_correct(prow, tname, tcode)

rag_df["correct"] = rag_df.apply(
    lambda r: pred_lookup.get((int(r["query_idx"]), r["trait"]), None), axis=1
)

case_rows = []
for case in range(top_k + 1):
    sub = rag_df[rag_df["n_match"] == case]
    cnt = len(sub)
    pct = round(100 * cnt / len(rag_df), 1)
    correct = sub["correct"].sum() if cnt > 0 else 0
    acc = round(100 * correct / cnt, 1) if cnt > 0 else None
    case_rows.append({"Case (# same-label)": case, "Count": cnt, "% total": pct,
                      "Correct preds": int(correct), "Accuracy (%)": acc})

case_df = pd.DataFrame(case_rows)
print(f"Case distribution  |  k={top_k}")
display(case_df)

In [ ]:
# ── per-trait case breakdown ──────────────────────────────────────────────────
per_trait_rows = []
for tname in TRAIT_NAMES:
    t = rag_df[rag_df["trait"] == tname]
    for case in range(top_k + 1):
        sub = t[t["n_match"] == case]
        cnt = len(sub)
        pct = round(100 * cnt / len(t), 1) if len(t) > 0 else 0
        correct = sub["correct"].sum() if cnt > 0 else 0
        acc = round(100 * correct / cnt, 1) if cnt > 0 else None
        per_trait_rows.append({"Trait": tname, "Case": case, "Count": cnt,
                                "% of trait": pct, "Correct preds": int(correct),
                                "Accuracy (%)": acc})

pt_df = pd.DataFrame(per_trait_rows)
display(pt_df)

## Step 6 — Visualisations

In [ ]:
# ── macro-F1 bar chart ────────────────────────────────────────────────────────
f1_vals = [f1_row[t] for t in TRAIT_NAMES]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(TRAIT_NAMES, f1_vals, color="steelblue", edgecolor="white")
ax.axhline(0.5, color="red", linestyle="--", linewidth=0.8, label="chance (0.5)")
ax.axhline(macro_avg, color="orange", linestyle="-.", linewidth=0.8, label=f"macro avg ({macro_avg:.3f})")
for bar, v in zip(bars, f1_vals):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.005, f"{v:.3f}",
            ha="center", va="bottom", fontsize=9)
ax.set_ylim(0, 1.0)
ax.set_ylabel("Macro-F1")
ax.set_title(f"Macro-F1 per trait — {prompt_mode}  (k={top_k})")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# ── bias chart: GT vs predicted-high ratio ────────────────────────────────────
x     = np.arange(len(TRAIT_NAMES))
width = 0.35
gt_vals   = [gt_ratios.get(t, 0)   for t in TRAIT_NAMES]
pred_vals = [pred_ratios.get(t, 0) for t in TRAIT_NAMES]

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(x - width/2, gt_vals,   width, label="Ground truth",   color="#2196F3", alpha=0.85)
ax.bar(x + width/2, pred_vals, width, label="Predicted high", color="#FF9800", alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(TRAIT_NAMES, rotation=15, ha="right")
ax.set_ylim(0, 1.0)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.set_ylabel("Fraction labelled 'high'")
ax.set_title(f"Predicted-high ratio vs. ground truth — {prompt_mode}")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── MMR / HR bar chart ────────────────────────────────────────────────────────
mmr_plot = mmr_df[mmr_df["Trait"] != "Macro avg"]
x     = np.arange(len(TRAIT_NAMES))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(x - width/2, mmr_plot["MMR"], width, label="MMR",      color="#4CAF50", alpha=0.85)
ax.bar(x + width/2, mmr_plot["HR"],  width, label="Hit-rate", color="#9C27B0", alpha=0.85)
ax.axhline(0.5, color="red", linestyle="--", linewidth=0.8, label="chance MMR")
for i, (mmr, hr) in enumerate(zip(mmr_plot["MMR"], mmr_plot["HR"])):
    ax.text(x[i] - width/2, mmr + 0.005, f"{mmr:.3f}", ha="center", va="bottom", fontsize=8)
    ax.text(x[i] + width/2, hr  + 0.005, f"{hr:.3f}",  ha="center", va="bottom", fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels(TRAIT_NAMES, rotation=15, ha="right")
ax.set_ylim(0, 1.1)
ax.set_ylabel("Score")
ax.set_title(f"Retrieval MMR & HR per trait — {prompt_mode}  (k={top_k})")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── case distribution + accuracy ─────────────────────────────────────────────
fig, ax1 = plt.subplots(figsize=(8, 4))
ax2 = ax1.twinx()

cases  = case_df["Case (# same-label)"].tolist()
counts = case_df["Count"].tolist()
accs   = [v if v is not None else 0 for v in case_df["Accuracy (%)"].tolist()]

ax1.bar(cases, counts, color="#42A5F5", alpha=0.7, label="Query count")
ax2.plot(cases, accs, "o-", color="#EF5350", linewidth=2, label="Accuracy (%)")

ax1.set_xlabel(f"# same-label retrieved (out of k={top_k})")
ax1.set_ylabel("Query count")
ax2.set_ylabel("Prediction accuracy (%)")
ax2.set_ylim(0, 110)
ax1.set_title(f"Case distribution & accuracy — {prompt_mode}  (k={top_k})")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

## Summary — copy-paste ready for thesis tables

In [ ]:
sep = "=" * 70
print(sep)
print(f"  RUN SUMMARY  |  mode={prompt_mode}  |  k={top_k}")
print(sep)

print("\n[tab:prompt-ablation row]")
cells = " & ".join(f"{f1_row[t]:.4f}" for t in TRAIT_NAMES)
print(f"  {prompt_mode} & {cells} & {macro_avg:.4f} \\\\")

print("\n[tab:bias row]")
cells = " & ".join(f"{pred_ratios.get(t, ''):.4f}" for t in TRAIT_NAMES)
print(f"  {prompt_mode} & {cells} \\\\")

print("\n[tab:retrieval-ablation  MMR/HR  (k=%d)]" % top_k)
for row in mmr_df[mmr_df["Trait"] != "Macro avg"].itertuples():
    print(f"  {row.Trait:20s}  MMR={row.MMR:.4f}  HR={row.HR:.4f}")
macro_row = mmr_df[mmr_df["Trait"] == "Macro avg"].iloc[0]
print(f"  {'Macro avg':20s}  MMR={macro_row['MMR']:.4f}  HR={macro_row['HR']:.4f}")
print(sep)